In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Create your project folders (Run this once to stay organized)
import os
PROJECT_PATH = '/content/drive/MyDrive/resume_screener'
os.makedirs(f"{PROJECT_PATH}/data", exist_ok=True)
os.makedirs(f"{PROJECT_PATH}/src", exist_ok=True)
os.makedirs(f"{PROJECT_PATH}/embeddings", exist_ok=True)

# 3. Install AI & File Processing Libraries
!pip install -q sentence-transformers datasets pdfplumber python-docx transformers torch
!python -m spacy download en_core_web_md -q

print("\n✅ Environment Ready!")


In [ ]:
# Phase 0: Data Generation
from datasets import load_dataset
import pandas as pd
import os

# 1. Define the path
DATA_DIR = '/content/drive/MyDrive/resume_screener/data'
os.makedirs(DATA_DIR, exist_ok=True)
SAVE_PATH = os.path.join(DATA_DIR, 'cs_resumes.csv')

# 2. Load the Resume-Atlas dataset from Hugging Face
print("Downloading dataset from Hugging Face...")
ds = load_dataset("ahmedheakl/resume-atlas", split="train")
df = ds.to_pandas()

# 3. Filter for CS-specific categories
cs_categories = [
    'Java Developer', 'Testing', 'SQL Developer', 'Network Security Engineer',
    'DotNet Developer', 'Web Designing', 'SAP Developer', 'Data Science',
    'ETL Developer', 'DevOps', 'Information Technology', 'Database',
    'Python Developer', 'React Developer', 'Blockchain'
]

cs_df = df[df['Category'].isin(cs_categories)].reset_index(drop=True)

# 4. Save to your Drive
cs_df.to_csv(SAVE_PATH, index=False)
print(f"✅ Successfully created {SAVE_PATH}")
print(f"Total CS resumes saved: {len(cs_df)}")

In [ ]:
%%writefile /content/drive/MyDrive/resume_screener/src/parser.py
import re

def clean_text(text):
    """
    Removes noise but keeps technical terms like C++ and .NET.
    """
    if not isinstance(text, str):
        return ""
    # Remove HTML tags and URLs
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    # Fix whitespace (very important for messy Hugging Face text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
%%writefile /content/drive/MyDrive/resume_screener/src/parser.py
import re
import pandas as pd
import pdfplumber
import docx

def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_resumes_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    df['cleaned_text'] = df['Text'].apply(clean_text)
    return df[df['cleaned_text'].str.len() > 50].reset_index(drop=True)

def load_single_resume(file_path):
    """Returns (raw_text, cleaned_text) for PDF or DOCX"""
    raw_text = ""
    if file_path.endswith('.pdf'):
        with pdfplumber.open(file_path) as pdf:
            raw_text = "\n".join([p.extract_text() for p in pdf.pages if p.extract_text()])
    elif file_path.endswith('.docx'):
        doc = docx.Document(file_path)
        raw_text = "\n".join([p.text for p in doc.paragraphs])

    if not raw_text.strip():
        return "Empty Resume", ""

    return raw_text, clean_text(raw_text)

In [ ]:
%%writefile /content/drive/MyDrive/resume_screener/src/embedder.py
from sentence_transformers import SentenceTransformer, util

# This model is small, fast, and perfect for semantic search.
model = SentenceTransformer('all-MiniLM-L6-v2')

def get_rankings(jd_text, resume_list):
    """
    Compares one Job Description to a list of resumes.
    Returns a list of similarity scores (0.0 to 1.0).
    """
    # Convert JD to a vector
    jd_vec = model.encode([jd_text], convert_to_numpy=True)

    # Convert all Resumes to vectors
    res_vecs = model.encode(resume_list, show_progress_bar=True, convert_to_numpy=True)

    # Calculate math similarity
    scores = util.cos_sim(jd_vec, res_vecs)[0].tolist()
    return scores

In [ ]:
%%writefile /content/drive/MyDrive/resume_screener/src/ner.py
import re
from transformers import pipeline

# Load AI
ai_worker = pipeline("ner", model="yashpwr/resume-ner-bert-v2", aggregation_strategy="simple")

NAME_BLACKLIST = ["Dxc", "Technology", "Curriculum", "Vitae", "Resume", "Solutions", "Services"]

TECH_SKILLS_DB = [
    "Python", "Java", "C++", "C#", "JavaScript", "React", "Node", "Angular",
    "SQL", "MongoDB", "AWS", "Docker", "Kubernetes", "Machine Learning",
    "Data Science", "Blockchain", "Solidity", "HTML", "CSS"
]

EDUCATION_KEYWORDS = {
    "Master's": ["mtech", "m.tech", "ms ", "m.s", "master", "mba", "msc"],
    "Bachelor's": ["btech", "b.tech", "be ", "b.e", "bachelor", "bsc"],
    "PhD": ["phd", "doctorate"]
}

def is_valid_name(name):
    name_lower = name.lower()
    return not any(word.lower() in name_lower for word in NAME_BLACKLIST)

def extract_all(cleaned_text, raw_text=None, candidate_id=None):
    input_text = raw_text if raw_text else cleaned_text

    # 1. Name
    name = "Unknown"
    results = ai_worker(input_text[:500])
    for res in results:
        if res['entity_group'] == 'NAME':
            temp_name = res['word'].title()
            if is_valid_name(temp_name) and len(temp_name) > 3:
                name = temp_name
                break

    if name == "Unknown" and candidate_id is not None:
        lines = [l.strip() for l in raw_text.split('\n') if len(l.strip()) > 2]
        name = lines[0].title() if lines else f"Candidate_{candidate_id + 101}"

    # 2. Skills
    ai_skills = {res['word'].title() for res in results if res['entity_group'] == 'SKILL'}
    found_tech = [s for s in TECH_SKILLS_DB if re.search(rf'\b{s}\b', cleaned_text, re.IGNORECASE)]
    final_skills = list(ai_skills.union(set(found_tech)))

    # 3. Education
    edu = "Not Specified"
    for level, keys in EDUCATION_KEYWORDS.items():
        if any(k in cleaned_text.lower() for k in keys):
            edu = level
            break

    # 4. Experience (Enhanced Regex)
    exp_match = re.search(r'(\d+)\s*(?:\+|-|to)?\s*(?:\d+)?\s*years?', cleaned_text, re.IGNORECASE)
    experience = f"{exp_match.group(1)} yrs" if exp_match else "Fresher"

    return {"name": name, "education": edu, "experience": experience, "skills": final_skills}

In [ ]:
import sys
from importlib import reload

# Ensure the path is correct
if '/content/drive/MyDrive/resume_screener/src' not in sys.path:
    sys.path.append('/content/drive/MyDrive/resume_screener/src')

In [ ]:
# FORCE RELOAD
import ner
reload(ner)
from ner import extract_all

# --- NOW RUN THE MAIN LOOP ---
import pandas as pd
from parser import load_resumes_from_csv
from embedder import get_rankings

DATA_PATH = '/content/drive/MyDrive/resume_screener/data/cs_resumes.csv'
df = load_resumes_from_csv(DATA_PATH)
subset = df.head(50).copy()

extracted_data = []
print("Extracting details...")

for idx, row in subset.iterrows():
    # Calling with explicit keyword arguments to be safe
    info = extract_all(cleaned_text=row['cleaned_text'], raw_text=row['Text'], candidate_id=idx)
    extracted_data.append(info)

# Merge and Rank
info_df = pd.DataFrame(extracted_data)
final_df = pd.concat([subset.reset_index(drop=True), info_df], axis=1)

job_query = "Looking for a Blockchain Developer with Solidity experience"
final_df['match_score'] = get_rankings(job_query, final_df['cleaned_text'].tolist())

# View Top 5
print(final_df[['name', 'education', 'skills', 'match_score']].sort_values('match_score', ascending=False).head(5))

In [ ]:
import sys
import pandas as pd
# Link to your Drive source folder
sys.path.append(f"{PROJECT_PATH}/src")
import parser
import ner
from parser import load_resumes_from_csv
from ner import extract_all
from embedder import get_rankings

# 1. Load Data
# (Assuming your csv is at: /content/drive/MyDrive/resume_screener/data/cs_resumes.csv)
DATA_PATH = f"{PROJECT_PATH}/data/cs_resumes.csv"
df = load_resumes_from_csv(DATA_PATH)

# 2. Process first 20 Resumes (Testing Batch)
subset = df.head(20).copy()
extracted_info = [extract_all(row['cleaned_text'], row['Text']) for _, row in subset.iterrows()]
info_df = pd.DataFrame(extracted_info)

# Merge results
final_df = pd.concat([subset, info_df], axis=1)

# 3. Rank against a Job Description
query_jd = "Looking for a Blockchain Developer with Solidity and Smart Contract skills"
final_df['match_score'] = get_rankings(query_jd, final_df['cleaned_text'].tolist())

# 4. View Results
top_results = final_df.sort_values(by='match_score', ascending=False)
print(top_results[['name', 'education', 'experience', 'match_score']].head(5))

In [ ]:
%%writefile /content/drive/MyDrive/resume_screener/app.py
import streamlit as st
import pandas as pd
import sys
import os
import re

sys.path.append('/content/drive/MyDrive/resume_screener/src')
from parser import load_single_resume
from ner import extract_all
from embedder import get_rankings

st.set_page_config(page_title="AI Resume Ranker Pro", layout="wide")
st.title("Resume Ranking Dashboard")

# --- SIDEBAR ---
st.sidebar.header("Job Configuration")
jd_input = st.sidebar.text_area("Target Job Description:", "Looking for an AI/ML Engineer...", height=150)

st.sidebar.subheader("Ranking Weights")
ai_w = st.sidebar.slider("AI Meaning Weight", 0.0, 1.0, 0.7)
kw_w = st.sidebar.slider("Keyword Weight", 0.0, 1.0, 0.3)

tab1, tab2 = st.tabs(["📂 Batch Upload & Rank", "👤 Single Profile Analysis"])

def get_kw_score(jd, res_text):
    jd_words = set(re.findall(r'\w+', jd.lower()))
    res_words = set(re.findall(r'\w+', res_text.lower()))
    tech_jd = {w for w in jd_words if len(w) > 3}
    if not tech_jd: return 0.5
    return len(tech_jd.intersection(res_words)) / len(tech_jd)

# --- TAB 1: BATCH ---
with tab1:
    st.header("Upload Multiple Resumes")
    files = st.file_uploader("Select PDFs or DOCX", type=["pdf", "docx"], accept_multiple_files=True, key="batch_upload")

    if st.button("Rank All Candidates") and files:
        results, cleaned_texts = [], []
        for i, f in enumerate(files):
            t_path = os.path.join("/content", f.name)
            with open(t_path, "wb") as file: file.write(f.getbuffer())
            raw, cleaned = load_single_resume(t_path)
            data = extract_all(cleaned, raw, i)
            results.append(data)
            cleaned_texts.append(cleaned)

        ai_scores = get_rankings(jd_input, cleaned_texts)
        final_list = []
        for i, score in enumerate(ai_scores):
            kw_s = get_kw_score(jd_input, cleaned_texts[i])
            combined = (score * ai_w) + (kw_s * kw_w)
            res = results[i]
            res['Match Score'] = f"{combined:.2%}"
            res['raw_score'] = combined
            final_list.append(res)

        df = pd.DataFrame(final_list).sort_values('raw_score', ascending=False)
        st.subheader("Final Rankings")
        st.dataframe(df[['name', 'Match Score', 'experience', 'education', 'skills']], use_container_width=True)

# --- TAB 2: SINGLE ---
with tab2:
    st.header("Detailed Candidate Analysis")
    single_file = st.file_uploader("Upload a single resume", type=["pdf", "docx"], key="single_upload")

    if single_file:
        t_path = os.path.join("/content", single_file.name)
        with open(t_path, "wb") as f: f.write(single_file.getbuffer())

        raw, cleaned = load_single_resume(t_path)
        res = extract_all(cleaned, raw)

        # Calculate scores
        ai_s = get_rankings(jd_input, [cleaned])[0]
        kw_s = get_kw_score(jd_input, cleaned)
        final_s = (ai_s * ai_w) + (kw_s * kw_w)

        # Layout
        col1, col2 = st.columns([2, 1])
        with col1:
            st.markdown(f"### Candidate: {res['name']}")
            st.write(f"**🏫 Education:** {res['education']}")
            st.write(f"**⏳ Experience:** {res['experience']}")
            st.write("**🛠 Skills Detected:**")
            st.write(", ".join(res['skills']) if res['skills'] else "None detected")

        with col2:
            st.metric("Final Match Score", f"{final_s:.2%}")
            st.progress(final_s)
            if final_s > 0.7: st.success("Strong Candidate")
            elif final_s > 0.4: st.warning("Potential Match")
            else: st.error("Low Relevance")

In [ ]:
%%writefile /content/drive/MyDrive/resume_screener/app.py
import streamlit as st
import pandas as pd
import sys
import os
import re

sys.path.append('/content/drive/MyDrive/resume_screener/src')
from parser import load_single_resume
from ner import extract_all
from embedder import get_rankings

st.set_page_config(page_title="AI Resume Ranker Pro", layout="wide")
st.title("Resume Ranking Dashboard")

# --- SIDEBAR ---
st.sidebar.header("Job Configuration")
jd_input = st.sidebar.text_area("Target Job Description:", "Looking for an AI/ML Engineer...", height=250)
st.sidebar.info("Ranking is currently set to a 1:1 balance of AI Semantic Meaning and Keyword Matching.")

AI_W = 1.0
KW_W = 1.0
TOTAL_W = AI_W + KW_W

tab1, tab2 = st.tabs(["📂 Batch Upload & Rank", "👤 Single Profile Analysis"])

def get_kw_score(jd, res_text):
    jd_words = set(re.findall(r'\w+', jd.lower()))
    res_words = set(re.findall(r'\w+', res_text.lower()))
    tech_jd = {w for w in jd_words if len(w) > 3}
    if not tech_jd: return 0.5
    return len(tech_jd.intersection(res_words)) / len(tech_jd)

# --- TAB 1: BATCH ---
with tab1:
    st.header("Upload Multiple Resumes")
    files = st.file_uploader("Select PDFs or DOCX", type=["pdf", "docx"], accept_multiple_files=True, key="batch_upload")

    if st.button("Rank All Candidates") and files:
        results, cleaned_texts = [], []
        for i, f in enumerate(files):
            t_path = os.path.join("/content", f.name)
            with open(t_path, "wb") as file: file.write(f.getbuffer())
            raw, cleaned = load_single_resume(t_path)
            data = extract_all(cleaned, raw, i) # Passing index i for name fallback
            results.append(data)
            cleaned_texts.append(cleaned)

        ai_scores = get_rankings(jd_input, cleaned_texts)
        final_list = []
        for i, score in enumerate(ai_scores):
            kw_s = get_kw_score(jd_input, cleaned_texts[i])
            combined = (score * (AI_W/TOTAL_W)) + (kw_s * (KW_W/TOTAL_W))
            res = results[i]
            res['Match Score'] = f"{combined:.2%}"
            res['raw_score'] = combined
            final_list.append(res)

        df = pd.DataFrame(final_list).sort_values('raw_score', ascending=False)
        st.subheader("Final Rankings")
        st.dataframe(df[['name', 'Match Score', 'experience', 'education', 'skills']], use_container_width=True)

# --- TAB 2: SINGLE ---
with tab2:
    st.header("Detailed Candidate Analysis")
    single_file = st.file_uploader("Upload a single resume", type=["pdf", "docx"], key="single_upload")

    if single_file:
        t_path = os.path.join("/content", single_file.name)
        with open(t_path, "wb") as f: f.write(single_file.getbuffer())

        raw, cleaned = load_single_resume(t_path)
        # FIX: Added 'candidate_id=0' here so the name fallback works exactly like Batch mode
        res = extract_all(cleaned, raw, candidate_id=0)

        ai_s = get_rankings(jd_input, [cleaned])[0]
        kw_s = get_kw_score(jd_input, cleaned)
        final_s = (ai_s * (AI_W/TOTAL_W)) + (kw_s * (KW_W/TOTAL_W))

        col1, col2 = st.columns([2, 1])
        with col1:
            st.markdown(f"### Candidate: {res['name']}")
            st.write(f"**🏫 Education:** {res['education']}")
            st.write(f"**⏳ Experience:** {res['experience']}")
            st.write("**🛠 Skills Detected:**")
            st.write(", ".join(res['skills']) if res['skills'] else "None detected")

        with col2:
            st.metric("Final Match Score", f"{final_s:.2%}")
            st.progress(final_s)

In [ ]:
!pip install -q pyngrok
!pip install -q streamlit pyngrok
import os
import time
from pyngrok import ngrok
from google.colab import userdata

# --- SETTINGS ---
AUTH_TOKEN = userdata.get('NGROK_TOKEN')
APP_PATH = "/content/drive/MyDrive/resume_screener/app.py"

# --- 1. KILL OLD PROCESSES ---
!pkill streamlit
!pkill ngrok
ngrok.kill()

# --- 2. START STREAMLIT ---
if os.path.exists(APP_PATH):
    # Running without 'nohup' for a second to catch immediate errors
    print("🚀 Starting Streamlit server...")
    os.system(f"nohup streamlit run {APP_PATH} --server.port 8501 &")

    print("⏳ Waiting 20 seconds for AI models (BERT/Sentence-Transformers) to load...")
    time.sleep(20)
    from pyngrok import ngrok
try:
    NGROK_TOKEN = userdata.get('NGROK_AUTH')
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(5000).public_url
    print(f"🚀 YOUR APP IS LIVE AT: {public_url}")
    print("Keep this cell running to access the URL.")
    !python app.py

except Exception as e:
    print(f"⚠️ Connection Error: {e}")
    print("Check if you added 'NGROK_AUTH' to your Colab Secrets.")
